# NHL Clean Shots EDA (2020-21 → 2024-25)

**Dataset:** [samuarg/nhl-clean-shots-data-2020-2021-to-2024-2025](https://www.kaggle.com/datasets/samuarg/nhl-clean-shots-data-2020-2021-to-2024-2025)

**Download first:** `python scripts/download_nhl_shots.py`

---
### Questions to explore
1. What does the shot data look like? (schema, size, nulls)
2. Shot volume by season, team, player
3. Shot type and danger zone breakdown
4. Shot distance / angle distributions
5. Goals vs. non-goals — what separates them? (xG foundation)
6. Game situation effects (even strength, PP, SH)
7. Goalie save % by shot type and zone

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('../data/raw/nhl_shots')

## 1. Load Data

In [ ]:
csv_files = sorted(DATA_DIR.glob('**/*.csv'))
print(f'Found {len(csv_files)} CSV file(s):')
for f in csv_files:
    size_mb = f.stat().st_size / 1_048_576
    print(f'  {f.name}  ({size_mb:.1f} MB)')

In [ ]:
# Load all CSVs — adjust if they're per-season files
dfs = [pd.read_csv(f, low_memory=False) for f in csv_files]
df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 2. Schema & Data Quality

In [ ]:
# Column types and null rates
quality = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(1),
    'n_unique': df.nunique()
})
quality

In [ ]:
df.describe(include='all').T

In [ ]:
# Peek at categorical columns
cat_cols = df.select_dtypes('object').columns.tolist()
for col in cat_cols[:10]:
    print(f'{col}: {df[col].value_counts().head(5).to_dict()}')

## 3. Column Mapping

Map column names to standard names used throughout the notebook. Adjust based on actual column names found above.

In [ ]:
# ── ADJUST THESE to match the actual column names in the dataset ──
COL = {
    'season':       'season',          # e.g. '2022-23'
    'game_id':      'gameId',
    'team':         'shootingTeam',
    'shooter':      'shooterName',
    'goalie':       'goalieNameForShot',
    'shot_type':    'shotType',
    'event':        'event',           # 'SHOT' or 'GOAL'
    'is_goal':      'goal',            # 0/1 or True/False
    'distance':     'shotDistance',
    'angle':        'shotAngle',
    'x':            'xCordAdjusted',
    'y':            'yCordAdjusted',
    'danger':       'shotGeneratedRebound',  # update if dataset has danger zone col
    'game_state':   'situation',       # 'ev', 'pp', 'sh', etc.
    'period':       'period',
}

# Validate — show any missing
missing = [v for v in COL.values() if v not in df.columns]
print('Columns not found in dataset (update COL dict above):', missing or 'None — all good!')

## 4. Shot Volume by Season

In [ ]:
season_col = COL['season']
goal_col   = COL['is_goal']

by_season = (
    df.groupby(season_col)
      .agg(shots=('game_id' if 'game_id' in df.columns else season_col, 'count'),
           goals=(goal_col, 'sum'))
      .assign(shooting_pct=lambda x: (x['goals'] / x['shots'] * 100).round(2))
)
by_season

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
by_season['shots'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Total Shots by Season')
axes[0].set_ylabel('Shot Count')
axes[0].tick_params(axis='x', rotation=30)

by_season['shooting_pct'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Shooting % by Season')
axes[1].set_ylabel('Shooting %')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Shot Type Breakdown

In [ ]:
shot_type_col = COL['shot_type']

shot_type_stats = (
    df.groupby(shot_type_col)
      .agg(shots=(goal_col, 'count'), goals=(goal_col, 'sum'))
      .assign(shooting_pct=lambda x: (x['goals'] / x['shots'] * 100).round(2))
      .sort_values('shots', ascending=False)
)
print(shot_type_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shot_type_stats['shots'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Shot Volume by Type')
axes[0].invert_yaxis()

shot_type_stats['shooting_pct'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Shooting % by Shot Type')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Shot Distance & Angle Distributions

In [ ]:
dist_col  = COL['distance']
angle_col = COL['angle']

if dist_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for label, mask, color in [
        ('No Goal', df[goal_col] == 0, 'steelblue'),
        ('Goal',    df[goal_col] == 1, 'coral'),
    ]:
        axes[0].hist(df.loc[mask, dist_col].dropna(), bins=60, alpha=0.6, label=label, color=color, density=True)

    axes[0].set_title('Shot Distance: Goals vs. Non-Goals')
    axes[0].set_xlabel('Distance (ft)')
    axes[0].legend()

    if angle_col in df.columns:
        for label, mask, color in [
            ('No Goal', df[goal_col] == 0, 'steelblue'),
            ('Goal',    df[goal_col] == 1, 'coral'),
        ]:
            axes[1].hist(df.loc[mask, angle_col].dropna(), bins=60, alpha=0.6, label=label, color=color, density=True)
        axes[1].set_title('Shot Angle: Goals vs. Non-Goals')
        axes[1].set_xlabel('Angle (°)')
        axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{dist_col}" not found — update COL dict')

## 7. Shot Map (Ice Surface Heatmap)

In [ ]:
x_col = COL['x']
y_col = COL['y']

if x_col in df.columns and y_col in df.columns:
    coords = df[[x_col, y_col, goal_col]].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    titles = ['All Shots', 'Goals Only']
    masks  = [slice(None), coords[goal_col] == 1]

    for ax, title, mask in zip(axes, titles, masks):
        subset = coords.loc[mask]
        ax.hexbin(subset[x_col], subset[y_col], gridsize=40, cmap='YlOrRd', mincnt=1)
        ax.set_xlim(-100, 100)
        ax.set_ylim(-42.5, 42.5)
        ax.set_title(title)
        ax.set_xlabel('x (ft)')
        ax.set_ylabel('y (ft)')
        # Net
        ax.scatter([89], [0], marker='D', color='black', s=80, zorder=5, label='Net')
        ax.legend()

    plt.suptitle('Shot Locations (Offensive Zone)', y=1.01, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f'Columns "{x_col}" / "{y_col}" not found — update COL dict')

## 8. Game Situation (Even Strength / Power Play / Short-Handed)

In [ ]:
state_col = COL['game_state']

if state_col in df.columns:
    state_stats = (
        df.groupby(state_col)
          .agg(shots=(goal_col, 'count'), goals=(goal_col, 'sum'))
          .assign(shooting_pct=lambda x: (x['goals'] / x['shots'] * 100).round(2))
          .sort_values('shots', ascending=False)
    )
    print(state_stats.to_string())

    state_stats['shooting_pct'].plot(kind='bar', color='mediumpurple', figsize=(10, 4))
    plt.title('Shooting % by Game Situation')
    plt.ylabel('Shooting %')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{state_col}" not found — update COL dict')

## 9. Top Shooters (by shot volume & goals)

In [ ]:
shooter_col = COL['shooter']

if shooter_col in df.columns:
    top_shooters = (
        df.groupby(shooter_col)
          .agg(shots=(goal_col, 'count'), goals=(goal_col, 'sum'))
          .assign(shooting_pct=lambda x: (x['goals'] / x['shots'] * 100).round(2))
          .sort_values('shots', ascending=False)
          .head(20)
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    top_shooters['shots'].plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Top 20 Shooters by Shot Volume')
    axes[0].invert_yaxis()

    top_shooters.sort_values('goals', ascending=False)['goals'].plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('Top 20 Shooters by Goals')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{shooter_col}" not found — update COL dict')

## 10. Goalie Save % by Shot Type

In [ ]:
goalie_col = COL['goalie']

if goalie_col in df.columns:
    goalie_stats = (
        df.groupby(goalie_col)
          .agg(shots_faced=(goal_col, 'count'), goals_against=(goal_col, 'sum'))
          .assign(save_pct=lambda x: ((1 - x['goals_against'] / x['shots_faced']) * 100).round(2))
          .query('shots_faced >= 500')   # min shots threshold
          .sort_values('save_pct', ascending=False)
    )

    top_goalies = goalie_stats.head(20)
    top_goalies['save_pct'].plot(kind='barh', color='teal', figsize=(12, 7))
    plt.title('Top 20 Goalies by Save % (min 500 shots faced)')
    plt.xlabel('Save %')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f'Column "{goalie_col}" not found — update COL dict')

## 11. xG Feature Preview (Goal Probability Inputs)

Checking correlation of key features with goal outcome — a preview of what an xG model would use.

In [ ]:
xg_features = [COL['distance'], COL['angle'], COL['shot_type'], COL['game_state']]
xg_features = [c for c in xg_features if c in df.columns]

# Numeric correlations
num_features = [c for c in xg_features if df[c].dtype in [np.float64, np.int64]]
if num_features:
    corr = df[num_features + [goal_col]].corr()[goal_col].drop(goal_col)
    corr.sort_values().plot(kind='barh', color='slateblue', figsize=(10, 4))
    plt.title('Feature Correlation with Goal (numeric only)')
    plt.axvline(0, color='black', linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Binned shooting % by distance
if COL['distance'] in df.columns:
    df['distance_bin'] = pd.cut(df[COL['distance']], bins=range(0, 110, 5))
    dist_goal_rate = df.groupby('distance_bin')[goal_col].mean() * 100
    dist_goal_rate.plot(kind='bar', color='darkorange', figsize=(14, 4))
    plt.title('Goal Rate by Shot Distance (5ft bins)')
    plt.ylabel('Goal %')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 12. Next Steps

Based on this EDA, good directions to pursue:

| Analysis | File |
|---|---|
| **xG Model** — logistic regression / XGBoost on distance + angle + shot type + situation | `predictive/xg_model.ipynb` |
| **Goalie evaluation** — xG against vs. actual goals against | `predictive/goalie_eval.ipynb` |
| **Team shot profile clustering** — K-Means on team shot mix | `predictive/team_clustering.ipynb` |
| **SQL analytics** — DuckDB queries over CSVs | `sql/nhl_shots.sql` |
| **Interactive shot map** — Plotly + ice rink overlay | `visualizations/shot_map.ipynb` |